# Universal Multi-Layer Probe Inference

Test notebook for running inference across ALL probes and ALL layers

In [2]:
import sys
from pathlib import Path

# Add parent directory to path to import from src
sys.path.insert(0, str(Path.cwd().parent))

from src.probes.universal_multi_layer_inference import UniversalMultiLayerInferenceEngine

## Initialize the Engine

This loads all 450 probes (45 actions × 10 layers)

In [4]:
probes_dir = Path.cwd().parent / "data" / "probes_binary"

engine = UniversalMultiLayerInferenceEngine(
    probes_base_dir=probes_dir,
    model_name="google/gemma-3-4b-it",
    device=None,
    layer_range=(15, 30),
    include_sentiment=True,
    sentiment_probes_dir=Path.cwd().parent / "data" / "sentiment" # Auto-detect
)

Detected compute device: Apple Metal Performance Shaders (MPS)
Initializing UniversalMultiLayerInferenceEngine...
  Probes base dir: /Users/ivanculo/Desktop/Projects/Cogni_map/brije/data/probes_binary
  Model: google/gemma-3-4b-it
  Device: mps
  Layer range: 15-30 (16 layers)
  Sentiment probes: /Users/ivanculo/Desktop/Projects/Cogni_map/brije/data/sentiment

Loading probes from all layers...
Loaded probe from /Users/ivanculo/Desktop/Projects/Cogni_map/brije/data/probes_binary/layer_15/probe_abstracting.pth
Loaded probe from /Users/ivanculo/Desktop/Projects/Cogni_map/brije/data/probes_binary/layer_15/probe_accepting.pth
Loaded probe from /Users/ivanculo/Desktop/Projects/Cogni_map/brije/data/probes_binary/layer_15/probe_analogical_thinking.pth
Loaded probe from /Users/ivanculo/Desktop/Projects/Cogni_map/brije/data/probes_binary/layer_15/probe_analyzing.pth
Loaded probe from /Users/ivanculo/Desktop/Projects/Cogni_map/brije/data/probes_binary/layer_15/probe_applying.pth
Loaded probe from

`torch_dtype` is deprecated! Use `dtype` instead!


Detected vision-language model. Loading text-only (skipping vision tower)...


Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.54s/it]



✓ Initialization complete!



## Test Text #1: Analytical Reasoning

In [5]:
text1 = "I'm thinking about how to solve this complex problem step by step"

print(f"Text: {text1}\n")
print("="*80)

Text: I'm thinking about how to solve this complex problem step by step



### Mode 1: Flat ranked list (top 20 across ALL layers)

In [6]:
preds = engine.predict_all(text1, threshold=0.00001, top_k=20)

print("Top 20 predictions across all layers:\n")
for i, pred in enumerate(preds, 1):
    marker = "✓" if pred.is_active else " "
    print(f"  {marker} {i:2d}. {pred.action_name:30s} (Layer {pred.layer:2d})  {pred.confidence:.4f}")

Top 20 predictions across all layers:

  ✓  1. sentiment                      (Layer  9)  5.3125
  ✓  2. sentiment                      (Layer 11)  -2.6562
  ✓  3. sentiment                      (Layer  7)  -1.1797
  ✓  4. analyzing                      (Layer 26)  1.0000
  ✓  5. understanding                  (Layer 27)  1.0000
  ✓  6. analyzing                      (Layer 29)  1.0000
  ✓  7. emotion_understanding          (Layer 29)  1.0000
  ✓  8. understanding                  (Layer 30)  1.0000
  ✓  9. divergent_thinking             (Layer 24)  0.9961
  ✓ 10. metacognitive_regulation       (Layer 18)  0.9883
  ✓ 11. analyzing                      (Layer 20)  0.7891
  ✓ 12. sentiment                      (Layer  6)  -0.6641
  ✓ 13. analyzing                      (Layer 19)  0.5859
  ✓ 14. sentiment                      (Layer 10)  -0.5430
  ✓ 15. sentiment                      (Layer  5)  0.4902
  ✓ 16. sentiment                      (Layer  3)  -0.3457
  ✓ 17. analyzing           

## Batch Testing: Multiple Texts

Add as many texts as you want to test in bulk!

In [17]:
from collections import Counter, defaultdict
import numpy as np

# Add your test strings here
test_strings = [
    "The quarterly numbers look... interesting. Revenue up 12%, but margins down 3%. Customer acquisition costs rising while retention rates plateau. Something doesn't add up here.",
    "What if we completely flipped the script? Instead of chasing the same customers everyone else wants, what about targeting the segment nobody's paying attention to?",
    "Last quarter's campaign... we spent $50K on social media ads, got 200 signups, but only 15 converted. That's a 7.5% conversion rate. Industry average is 12%. We're bleeding money.",
    "That client meeting keeps replaying in my head. Sarah said 'the integration feels clunky' and I brushed it off. Now three clients have mentioned the same thing. I should have listened.",
    "My brain is scattered. Need to organize this mess: finish the Q4 budget review, prep for tomorrow's board meeting, and draft the hiring plan for next quarter. Otherwise I'll forget something crucial.",
    "Where are we on the Johnson account? Last I heard, legal was reviewing the contract. Marketing said they'd have the campaign ready by Friday. Finance needs the numbers by end of week. Everything's converging.",
    "Client feedback from Project Alpha, user research from Beta, and market analysis from Gamma. All pointing in different directions. There's a pattern here I'm not seeing yet.",
    "Option A: expand to Europe, higher risk but potentially 40% revenue growth. Option B: focus on domestic market, safer but maybe 15% growth. Both have merit. Both have downsides.",
    "The website keeps crashing during peak hours. Server logs show increased traffic, but that shouldn't cause failures. There's something else going on.",
    "The interns are lost. I'm throwing terms like 'conversion funnel' and 'attribution modeling' at them. They need the basics first - what we're trying to achieve and why.",
    "This product launch strategy feels incomplete. Maybe I should bounce ideas off the team. Fresh perspectives could reveal blind spots I'm missing.",
    "I've been assuming our target demographic is 25-35 year olds. But what if that's wrong? What if I'm basing decisions on outdated assumptions?",
    "This dashboard is overwhelming. Revenue charts, user engagement metrics, conversion rates, churn analysis. Too much noise. Need to focus on what actually matters.",
    "The current approach isn't working. Users aren't engaging with the new feature. Maybe we need to pivot. Try a different angle entirely.",
    "These customer segments look similar on paper - both tech-savvy, both high income. But their behavior patterns are completely different. What am I missing?",
    "If we launch in Q2 instead of Q1, we'd have more time for testing. But competitors might beat us to market. If we rush Q1, we risk bugs. If we wait, we risk irrelevance.",
    "The manager's email was vague: 'streamline the process.' What does that mean exactly? Reduce steps? Automate tasks? Cut costs? Need to clarify before I act.",
    "I'm recommending we increase the marketing budget by 30%. But why? Because last quarter's campaign worked? Because competitors are spending more? Need solid reasoning.",
    "Sally flagged that our pricing model doesn't account for seasonal fluctuations. She's right. Our revenue projections assume steady demand year-round. That's unrealistic.",
    "The project timeline is chaotic. Phase 1 should inform Phase 2, which should inform Phase 3. But everything's happening simultaneously. Need to map out dependencies.",
    "This market research feels biased. The methodology seems sound, but the conclusions feel predetermined. Like they found what they were looking for.",
    "The correlation between social media engagement and sales is strong. But that doesn't mean social media causes sales. Could be reverse causation, or a third factor entirely.",
    "Let's test this hypothesis: if our target users really want this feature, they'll use it within the first week. If not, we'll know it's not solving a real problem.",
    "Both theories explain the data well. Theory A focuses on user behavior, Theory B on market conditions. They're not mutually exclusive, but they emphasize different factors.",
    "This industry report cites impressive statistics, but I don't recognize the research firm. Need to verify their credibility before I base any decisions on their findings.",
    "Today's priorities are overwhelming. The client presentation, the budget review, the team meeting, the product demo. Can't do everything. Need to pick what's truly urgent.",
    "The alternative approach might be better. Current method is familiar, but the new one could be more efficient. Should we compare them side by side before deciding?",
    "Let me explain this simply: we're not making money because we're spending more to acquire customers than we earn from them. Like buying a $10 item for $15.",
    "I remember being overwhelmed by all these metrics and KPIs when I started. Jamie looks lost in the same way. Maybe I can help them understand what actually matters.",
    "Sitting here watching people interact with our app. Some scroll quickly, others pause and tap. Some get frustrated and leave. Others seem to find what they need. Patterns emerging."
]

threshold = 0.5
display_threshold = 0.5  # Only show actions with confidence >= 0.001

print(f"Processing {len(test_strings)} texts...\n")
print("="*80)

for i, text in enumerate(test_strings, 1):
    print(f"\n[{i}/{len(test_strings)}] Text:")
    print(f'"{text}"')
    print("-"*80)
    
    # Get predictions organized by layer
    layer_preds = engine.predict_by_layer(text, threshold=threshold)
    
    if layer_preds:
        # Display results for each layer 
        for layer in sorted(layer_preds.keys()):
            preds = layer_preds.get(layer, [])
            if preds:
                # Filter by display threshold and format actions with confidences
                filtered_preds = [p for p in preds if p.confidence >= display_threshold]
                if filtered_preds:
                    actions_str = ", ".join([f"{p.action_name}({p.confidence:.3f})" for p in filtered_preds])
                    print(f"  Layer {layer:2d}: {actions_str}")
    else:
        print("  No predictions above threshold")

print(f"\n{'='*80}")
print(f"Batch testing complete: {len(test_strings)} texts processed")

Processing 30 texts...


[1/30] Text:
"The quarterly numbers look... interesting. Revenue up 12%, but margins down 3%. Customer acquisition costs rising while retention rates plateau. Something doesn't add up here."
--------------------------------------------------------------------------------
  Layer  8: sentiment(1.586)
  Layer 16: hypothesis_generation(1.000), evaluating(0.797), emotion_understanding(0.680)
  Layer 17: hypothesis_generation(1.000)
  Layer 18: noticing(1.000), evaluating(0.930)
  Layer 19: hypothesis_generation(1.000), noticing(1.000)
  Layer 20: noticing(1.000)
  Layer 22: analyzing(1.000), hypothesis_generation(1.000), noticing(1.000)
  Layer 23: hypothesis_generation(1.000), noticing(1.000)
  Layer 24: analyzing(1.000), noticing(1.000)
  Layer 25: analyzing(1.000), noticing(1.000)
  Layer 26: analyzing(1.000), noticing(1.000)
  Layer 27: analyzing(1.000), hypothesis_generation(1.000), noticing(1.000), understanding(0.992)
  Layer 28: analyzing(1.000), evaluating